# 03 — MLP Translation Head Training

This notebook trains the MLP translation head that maps frozen Concerto features → CLIP text embedding space for semantic segmentation on S3DIS Area 5.

**Prerequisites:** Feature extraction (notebook 02) must be complete — the `features/s3dis_area5/` directory should contain 68 `.npz` files (~31 GB).

**Runtime:** ~30–60 min on Colab T4.

### 1. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'dev/enc-mlp'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}


In [ ]:
%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

In [ ]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto 2>/dev/null || echo 'Concerto already cloned'

### 2. Symlink Data & Checkpoints

In [ ]:
# Symlink Drive data
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features

In [ ]:
# Verify extracted features exist
!echo '--- Feature files ---'
!ls features/s3dis_area5/*.npz | wc -l
!echo 'files found'
!du -sh features/s3dis_area5/

### 3. Setup Environment

In [ ]:
%cd /content/Deep_learning_project/notebooks/

In [ ]:
!uv python install 3.10.12
!uv sync --python 3.10.12

In [ ]:
# Setup HF token for open_clip
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

### 4. (Optional) Copy features to local SSD for faster I/O
Google Drive I/O is slow. Copying the features to the Colab local disk speeds up training significantly.

Skip this cell if features are already local or you're low on disk space.

In [ ]:
%%time
import shutil, os

LOCAL_FEATURES = '/content/features_local/s3dis_area5'
DRIVE_FEATURES = '/content/Deep_learning_project/features/s3dis_area5'

if not os.path.exists(LOCAL_FEATURES):
    os.makedirs(LOCAL_FEATURES, exist_ok=True)
    print(f'Copying features from Drive to local SSD...')
    !cp -v {DRIVE_FEATURES}/*.npz {LOCAL_FEATURES}/
    print('Done!')
else:
    print(f'Local features already exist at {LOCAL_FEATURES}')

!ls {LOCAL_FEATURES}/*.npz | wc -l
!echo 'feature files on local SSD'

### 5. Update config to point to local features

If you copied features to local SSD in step 4, we patch the config to use the faster local path.

In [ ]:
import yaml

CONFIG_PATH = '/content/Deep_learning_project/configs/train_mlp_s3dis.yaml'
LOCAL_FEATURES = '/content/features_local/s3dis_area5'

with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

# Use local SSD path if available, otherwise keep Drive path
import os
if os.path.exists(LOCAL_FEATURES):
    cfg['data']['train_features_path'] = LOCAL_FEATURES
    print(f'Using LOCAL features: {LOCAL_FEATURES}')
else:
    print(f'Using DRIVE features: {cfg["data"]["train_features_path"]}')

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print('\n--- Current config ---')
!cat {CONFIG_PATH}

### 6. Run Training

This runs the MLP translation head training with:
- **Hybrid loss** (MSE + cosine) — aligns training with cosine-similarity evaluation
- **30 epochs** with cosine LR annealing
- **15% holdout validation** stratified by room type
- **torch.compile** for ~5-10% speedup
- **100% of Area 5 rooms** (no fraction sampling)

Estimated time: **30–60 min on T4**.

In [ ]:
%%time
# Train the MLP translation head
!PYTHONPATH=/content/Deep_learning_project:/content/Concerto \
  uv run python /content/Deep_learning_project/src/train.py \
  --config /content/Deep_learning_project/configs/train_mlp_s3dis.yaml

### 7. Plot Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt

METRICS_PATH = '/content/Deep_learning_project/checkpoints/mlp_s3dis/history.json'

with open(METRICS_PATH, 'r') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss = [h['val_loss'] for h in history if h['val_loss'] is not None]
val_epochs = [h['epoch'] for h in history if h['val_loss'] is not None]
lrs = [h.get('lr', None) for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(epochs, train_loss, label='Train Loss', linewidth=2)
if val_loss:
    axes[0].plot(val_epochs, val_loss, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LR plot
if lrs[0] is not None:
    axes[1].plot(epochs, lrs, linewidth=2, color='green')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Learning Rate')
    axes[1].set_title('Learning Rate Schedule')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/Deep_learning_project/checkpoints/mlp_s3dis/training_curves.png', dpi=150)
plt.show()

print(f"\nFinal train loss: {train_loss[-1]:.6f}")
if val_loss:
    print(f"Final val loss:   {val_loss[-1]:.6f}")
    print(f"Best val loss:    {min(val_loss):.6f}")

### 8. Evaluate (OA & mIoU)

In [ ]:
%%time
# Evaluate using the best checkpoint
CHECKPOINT = '/content/Deep_learning_project/checkpoints/mlp_s3dis/best_model.pth'
FEATURES_DIR = '/content/Deep_learning_project/features/s3dis_area5'

# Use local features if available
import os
if os.path.exists('/content/features_local/s3dis_area5'):
    FEATURES_DIR = '/content/features_local/s3dis_area5'

!PYTHONPATH=/content/Deep_learning_project:/content/Concerto \
  uv run python /content/Deep_learning_project/src/evaluate.py \
  --features_dir {FEATURES_DIR} \
  --checkpoint {CHECKPOINT} \
  --clip_model ViT-B-32 \
  --clip_pretrained openai

### 9. Copy checkpoints back to Drive

Make sure the trained model is safely stored on Drive.

In [ ]:
!mkdir -p /content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/
!cp -v /content/Deep_learning_project/checkpoints/mlp_s3dis/*.pth \
       /content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/
!cp -v /content/Deep_learning_project/checkpoints/mlp_s3dis/history.json \
       /content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/
!cp -v /content/Deep_learning_project/checkpoints/mlp_s3dis/training_curves.png \
       /content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/ 2>/dev/null || true
print('Checkpoints backed up to Drive!')